# Model 1: Transfer Learning with VGG16 for Lego Brick Classification

## Aim

The aim of this experiment is to apply transfer learning using the pre-trained VGG16 network to classify Lego brick images into five categories. VGG16 was originally trained on the ImageNet dataset. In Model 1, only the final 1000-class dense output layer is removed and replaced with a new dense layer containing 5 output neurons. All earlier layers are frozen so that only the replacement output layer is trained.

In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import zipfile
import shutil
import random
from pathlib import Path

## Dataset Extraction

The dataset is stored as a zip file and is extracted into the Colab runtime before preprocessing and splitting. The class folders are labelled 0, 1, 2, 3 and 4.

In [ ]:
zip_path = "small_Lego.zip"
extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully")
print(os.listdir(extract_path))

## Check Dataset Structure

The extracted dataset is checked to confirm that it contains five class folders. These folders will be used as the class labels by Keras.

In [ ]:
base_data_dir = "dataset/SmallLego"  
print(os.listdir(base_data_dir))

## Train, Validation and Test Split

The assignment requires the dataset to be split into training, validation and test sets using a 60/20/20 ratio. The same split should be used for both models so that the results can be compared fairly.

In [ ]:
random.seed(42)

source_dir = Path(base_data_dir)
split_root = Path("split_data")

train_dir = split_root / "train"
val_dir = split_root / "val"
test_dir = split_root / "test"

# remove old split if it exists
if split_root.exists():
    shutil.rmtree(split_root)

# create folders
for split in [train_dir, val_dir, test_dir]:
    split.mkdir(parents=True, exist_ok=True)

class_names = sorted([d.name for d in source_dir.iterdir() if d.is_dir()])
print("Classes:", class_names)

for class_name in class_names:
    class_source = source_dir / class_name
    images = [f for f in class_source.iterdir() if f.is_file()]
    random.shuffle(images)

    total = len(images)
    train_count = int(total * 0.6)
    val_count = int(total * 0.2)
    test_count = total - train_count - val_count

    class_train = train_dir / class_name
    class_val = val_dir / class_name
    class_test = test_dir / class_name

    class_train.mkdir(parents=True, exist_ok=True)
    class_val.mkdir(parents=True, exist_ok=True)
    class_test.mkdir(parents=True, exist_ok=True)

    train_files = images[:train_count]
    val_files = images[train_count:train_count + val_count]
    test_files = images[train_count + val_count:]

    for f in train_files:
        shutil.copy(f, class_train / f.name)

    for f in val_files:
        shutil.copy(f, class_val / f.name)

    for f in test_files:
        shutil.copy(f, class_test / f.name)

    print(f"Class {class_name}: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

## Pre-processing

The images are resized to 224 × 224 × 3, which is the input size required by VGG16. Preprocessing uses the standard VGG16 preprocessing function, which applies mean subtraction based on the ImageNet training procedure. No additional rescaling to the range 0 to 1 is applied.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
batch_size = 32
img_size = (224, 224)

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

## Model 1 Architecture

Model 1 retains the original VGG16 dense layers and replaces only the final 1000-class output layer with a new 5-class output layer. All earlier layers are frozen. This means that the model keeps the learned ImageNet feature representation and only learns a new classifier for the Lego brick categories.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense

In [ ]:
base_model = VGG16(weights='imagenet', include_top=True)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.layers[-2].output
output = Dense(5, activation='softmax')(x)

model1 = Model(inputs=base_model.input, outputs=output)

model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Model Summary

The model summary is shown below to confirm that the architecture matches the required specification. Model 1 should have approximately 134 million parameters in total, with only about 20,485 trainable parameters.

In [ ]:
model1.summary()

## Runtime Note

A GPU runtime is recommended for this experiment because training VGG16 on CPU is significantly slower.

## Training Setup

The model is trained for 10 epochs using categorical cross-entropy loss and the Adam optimiser. Training and validation accuracy and loss are recorded after each epoch in order to analyse learning behaviour.

In [ ]:
history1 = model1.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
model1.save("model1.h5")
print("Model saved!")

## Training Curves

The following graphs show training and validation accuracy and loss across 10 epochs. These graphs help to show whether the model is learning effectively and whether overfitting may be occurring.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history1.history['accuracy'], label='Training Accuracy')
plt.plot(history1.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model 1 Training and Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history1.history['loss'], label='Training Loss')
plt.plot(history1.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model 1 Training and Validation Loss')
plt.legend()
plt.show()

## Interpretation of the Training Curves

The training and validation curves are used to assess the behaviour of the model during learning. If training accuracy increases while validation accuracy also improves, this suggests that the model is learning useful patterns. If the training performance continues to improve while validation performance levels off or worsens, this may indicate overfitting.

## Test Set Evaluation

After 10 epochs, the final model is evaluated on the test set. The test accuracy provides a single summary value for performance on previously unseen data.

In [ ]:
test_loss, test_accuracy = model1.evaluate(test_data)
print("Model 1 Test Loss:", test_loss)
print("Model 1 Test Accuracy:", test_accuracy)

## Confusion Matrix

A 5 × 5 confusion matrix is used to show how predictions are distributed across the five classes. This provides more detailed information than accuracy alone by showing which classes are most often confused with one another.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
pred_probs = model1.predict(test_data)
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = test_data.classes

cm = confusion_matrix(true_labels, pred_labels)
print(cm)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(test_data.class_indices.keys()))
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title("Model 1 Confusion Matrix")
plt.show()

## Conclusion

Model 1 applies transfer learning by preserving almost the entire VGG16 architecture learned from ImageNet and replacing only the final classifier with a new 5-class output layer. This allows the model to adapt to Lego brick classification while training only a small number of parameters.